# SQL Analysis

This notebook is for running SQL-based exploration on the cleaned TMDB data.

In [15]:
# Reel Insights: SQL Analysis

## 1. Database Connection and Setup

import mysql.connector

# Create connection to the MySQL database using the provided credentials.
connection = mysql.connector.connect(
    host='gateway01.ap-southeast-1.prod.aws.tidbcloud.com',
    user='3YtRASBfEpfD1yv.root',
    password='tbg0bYlqolNYpd8Y',
    database='reel_insights'
)

print("Connection status:", connection.is_connected())

Connection status: True


In [13]:
# cursor.fetchall()
# cursor.execute()
# cursor.fetchone()

create_tables_sql = [
    """
    CREATE TABLE movies (
        movie_id INT PRIMARY KEY,
        title VARCHAR(255),
        original_language VARCHAR(10),
        release_date DATE,
        runtime INT,
        budget BIGINT,
        revenue BIGINT,
        popularity FLOAT,
        vote_average FLOAT,
        vote_count INT,
        status VARCHAR(50),
        overview TEXT
    )
    """,
    """
    CREATE TABLE genres (
        genre_id INT PRIMARY KEY,
        genre_name VARCHAR(50)
    )
    """,
    """
    CREATE TABLE movie_genres (
        movie_id INT,
        genre_id INT,
        PRIMARY KEY (movie_id, genre_id),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
        FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
    )
    """,
    """
    CREATE TABLE `cast` (
        cast_row_id INT PRIMARY KEY,
        movie_id INT,
        person_id INT,
        actor_name VARCHAR(255),
        character_name VARCHAR(255),
        cast_order INT,
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    )
    """,
    """
    CREATE TABLE crew (
        crew_row_id INT PRIMARY KEY,
        movie_id INT,
        person_id INT,
        person_name VARCHAR(255),
        job VARCHAR(100),
        department VARCHAR(100),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    )
    """,
    """
    CREATE TABLE movie_keywords (
        movie_id INT,
        keyword_id INT,
        keyword_name VARCHAR(100),
        PRIMARY KEY (movie_id, keyword_id),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    )
    """
]

cursor = connection.cursor()

for sql in create_tables_sql:
    cursor.execute(sql)

connection.commit()



In [ ]:
cursor = connection.cursor()

cursor.execute("SELECT DATABASE()")
print("Current database:", cursor.fetchone())

cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

print("Existing tables:", tables)

In [5]:
import os

print(os.listdir("C:\\Users\\acer\\Documents\\TMDB Project\\ReelInsights\\outputs"))

['cast_repeated_keys.csv', 'cleaned_cast.csv', 'cleaned_crew.csv', 'cleaned_movies.csv', 'cleaned_movies_data.csv', 'cleaned_movie_genres.csv', 'cleaned_movie_keywords.csv', 'crew_repeated_keys.csv', 'dataset_overview.csv', 'duplicate_summary.csv', 'key_quality_report.csv', 'missing_values.csv', 'movies_repeated_keys.csv', 'movie_genres_repeated_keys.csv', 'movie_keywords_repeated_keys.csv', 'relationship_report.csv']


In [ ]:
# Inserting the data into the respective tables in the database
# Movies DataFrame into the movies table

import pandas as pd

movies_df = pd.read_csv("C:\\Users\\acer\\Documents\\TMDB Project\\ReelInsights\\outputs\\cleaned_movies_data.csv")

insert_movies_query = """
INSERT INTO movies (movie_id, title, original_language, release_date, runtime, budget, revenue, 
popularity, vote_average, vote_count, status, overview)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor = connection.cursor()

rows = [
    tuple(row)
    for row in movies_df[["movie_id", "title", "original_language", "release_date", "runtime", "budget", "revenue",
                          "popularity", "vote_average", "vote_count", "status", "overview"]].values
]

cursor.executemany(insert_movies_query, rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM movies")
print("Number of rows in movies table:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM movies LIMIT 5")
for row in cursor.fetchall():
    print(row)

In [8]:
import pandas as pd

genres_df = pd.read_csv("C:\\Users\\acer\\Documents\\TMDB Project\\ReelInsights\\data\\genres.csv")

insert_genres = """
INSERT INTO genres (genre_id, genre_name)
VALUES (%s, %s)
"""

cursor = connection.cursor()

rows = [
    tuple(row)
    for row in genres_df[["genre_id", "genre_name"]].values
]

cursor.executemany(insert_genres, rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM genres")
print("Number of rows in movies table:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM genres LIMIT 5")
for row in cursor.fetchall():
    print(row)

Number of rows in movies table: 19
(12, 'Adventure')
(14, 'Fantasy')
(16, 'Animation')
(18, 'Drama')
(27, 'Horror')


In [4]:
import pandas as pd

movies_genres_df = pd.read_csv("C:\\Users\\acer\\Documents\\TMDB Project\\ReelInsights\\outputs\\cleaned_movie_genres.csv")

insert_movies_genres = """
INSERT INTO movie_genres(movie_id, genre_id)
VALUES(%s,%s)
"""
cursor = connection.cursor()

rows = [
    (int(row.movie_id), int(row.genre_id))
    for row in movies_genres_df.itertuples(index=False)
]


cursor.executemany(insert_movies_genres, rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM movie_genres")
print("Number of rows in movie_genres table:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM movie_genres LIMIT 5")
for row in cursor.fetchall():
    print(row)

Number of rows in movie_genres table: 6878
(11, 12)
(12, 12)
(18, 12)
(22, 12)
(58, 12)


In [11]:
import pandas as pd

cast_df = pd.read_csv("C:\\Users\\acer\\Documents\\TMDB Project\\ReelInsights\\outputs\\cleaned_cast.csv")

insert_cast = """
INSERT INTO cast (cast_row_id, movie_id, person_id,
actor_name, character_name, cast_order)
VALUES (%s, %s, %s, %s, %s, %s)
"""

rows =  [
    tuple(row)
    for row in cast_df[["cast_row_id", "movie_id", "person_id", "actor_name", "character_name", "cast_order"]].values
]

cursor = connection.cursor()
cursor.executemany(insert_cast,rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM cast")
print("Number of rows in cast table:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM cast LIMIT 5")
for row in cursor.fetchall():
    print(row)


Number of rows in cast table: 24902
(1, 157336, 10297, 'Matthew McConaughey', 'Cooper', 0)
(2, 157336, 1813, 'Anne Hathaway', 'Brand', 1)
(3, 157336, 3895, 'Michael Caine', 'Professor Brand', 2)
(4, 157336, 83002, 'Jessica Chastain', 'Murph', 3)
(5, 157336, 1893, 'Casey Affleck', 'Tom', 4)


In [14]:
import pandas as pd

crew_df = pd.read_csv("C:\\Users\\acer\\Documents\\TMDB Project\\ReelInsights\\outputs\\cleaned_crew.csv")

insert_crew = """
INSERT INTO crew (crew_row_id, movie_id, person_id,
person_name, job, department)
VALUES(%s, %s, %s, %s, %s, %s)
"""

rows = [
    tuple(row)
    for row in crew_df[["crew_row_id", "movie_id", "person_id",
"person_name", "job", "department"]].values
]

cursor = connection.cursor()
cursor.executemany(insert_crew, rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM crew")
print("Number of rows in cast table:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM crew LIMIT 5")
for row in cursor.fetchall():
    print(row)

Number of rows in cast table: 7289
(1, 157336, 527, 'Jonathan Nolan', 'Writer', 'Writing')
(2, 157336, 525, 'Christopher Nolan', 'Writer', 'Writing')
(3, 157336, 525, 'Christopher Nolan', 'Director', 'Directing')
(4, 27205, 525, 'Christopher Nolan', 'Writer', 'Writing')
(5, 27205, 525, 'Christopher Nolan', 'Director', 'Directing')


In [16]:
import pandas as pd

movies_keywords_df = pd.read_csv("C:\\Users\\acer\\Documents\\TMDB Project\\ReelInsights\\outputs\\cleaned_movie_keywords.csv")

insert_movies_keywords = """
INSERT INTO movie_keywords (
            movie_id, keyword_id, keyword_name
        ) VALUES (%s, %s, %s)
"""

rows = [
    tuple(row)
    for row in movies_keywords_df[["movie_id", "keyword_id", "keyword_name"]].values
]

cursor = connection.cursor()
cursor.executemany(insert_movies_keywords, rows)
connection.commit()

cursor.execute("SELECT COUNT(*) FROM movie_keywords")
print("Number of rows in movie_keywords table:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM movie_keywords LIMIT 5")
for row in cursor.fetchall():
    print(row)

Number of rows in movie_keywords table: 41681
(5, 612, 'hotel')
(5, 613, "new year's eve")
(5, 616, 'witch')
(5, 622, 'bet')
(5, 922, 'hotel room')
